# Transformation - Combining flight data and Timezones

In [1]:
import sys
from pathlib import Path

project_root = "/notebooks/ML/Flight Delay Prediction Project/ELT"
sys.path.append(str(project_root))

In [2]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pandas as pd

In [3]:
from src.utils.spark_session import get_spark_session
from src.load.flight_data import *
from src.load.timezones import *

In [4]:
pd.set_option('display.max_columns', None)

In [ ]:
spark = get_spark_session()

In [6]:
s3_key_fd = "cleaned/Flight_Delay_Prediction_Datasets/flight_data/"
s3_key_tz = "bronze/Flight_Delay_Prediction_Datasets/timezones/airport_timezones.parquet"

fd_df = get_flight_lake_dataset(spark, s3_key_fd)
tz_df = get_tz_lake_dataset(spark, s3_key_tz)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

In [7]:
fd_df.limit(5).toPandas()

/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,12:05,11:59,-6.0,0.0,0.0,-1,1200-1259,18.0,12:17,14:36,9.0,15:11,14:45,-26.0,0.0,0.0,-2,1500-1559,366.0,319.0,2611.0,11,0.0,0.0,0.0,0.0,0.0
1,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,15:20,15:38,18.0,18.0,1.0,1,1500-1559,11.0,15:49,23:44,9.0,23:54,23:53,-1.0,0.0,0.0,-1,2300-2359,334.0,295.0,2475.0,10,0.0,0.0,0.0,0.0,0.0
2,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,18:07,19:01,54.0,54.0,1.0,3,1800-1859,17.0,19:18,21:34,13.0,21:21,21:47,26.0,26.0,1.0,1,2100-2159,374.0,316.0,2611.0,11,6.0,0.0,0.0,0.0,20.0
3,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,10721,BOS,"Boston, MA",MA,08:35,09:09,34.0,34.0,1.0,2,0800-0859,17.0,09:26,17:46,12.0,17:17,17:58,41.0,41.0,1.0,2,1700-1759,342.0,320.0,2611.0,11,34.0,0.0,7.0,0.0,0.0
4,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,22:52,22:47,-5.0,0.0,0.0,-1,2200-2259,19.0,23:06,06:57,15.0,07:29,07:12,-17.0,0.0,0.0,-2,0700-0759,337.0,291.0,2475.0,10,0.0,0.0,0.0,0.0,0.0


In [8]:
tz_df.limit(5).toPandas()

/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,iata_code,ident,name,municipality,latitude_deg,longitude_deg,timezone,__index_level_0__
0,UTK,03N,Utirik Airport,Utirik Island,11.222219,169.851429,Pacific/Majuro,204
1,OCA,07FA,Ocean Reef Club Airport,Key Largo,25.325399,-80.274803,America/New_York,412
2,CSE,0CO2,Crested Butte Airpark,Crested Butte,38.851918,-106.928341,America/Denver,635
3,CUS,0NM0,Columbus Airport,Columbus,31.823898,-107.629924,America/Denver,893
4,JCY,0TE7,LBJ Ranch Airport,Stonewall,30.251801,-98.622498,America/Chicago,990


## Perform JOIN operation

In [9]:
from src.transform.combine_timezones import *

In [10]:
joined_df = combine_flights_timezones(spark, fd_df, tz_df)

## Validate data

In [11]:
joined_df.limit(5).toPandas()

/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,origin_timezone,destination_timezone
0,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,12:05,11:59,-6.0,0.0,0.0,-1,1200-1259,18.0,12:17,14:36,9.0,15:11,14:45,-26.0,0.0,0.0,-2,1500-1559,366.0,319.0,2611.0,11,0.0,0.0,0.0,0.0,0.0,America/New_York,America/Los_Angeles
1,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,15:20,15:38,18.0,18.0,1.0,1,1500-1559,11.0,15:49,23:44,9.0,23:54,23:53,-1.0,0.0,0.0,-1,2300-2359,334.0,295.0,2475.0,10,0.0,0.0,0.0,0.0,0.0,America/Los_Angeles,America/New_York
2,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,18:07,19:01,54.0,54.0,1.0,3,1800-1859,17.0,19:18,21:34,13.0,21:21,21:47,26.0,26.0,1.0,1,2100-2159,374.0,316.0,2611.0,11,6.0,0.0,0.0,0.0,20.0,America/New_York,America/Los_Angeles
3,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,10721,BOS,"Boston, MA",MA,08:35,09:09,34.0,34.0,1.0,2,0800-0859,17.0,09:26,17:46,12.0,17:17,17:58,41.0,41.0,1.0,2,1700-1759,342.0,320.0,2611.0,11,34.0,0.0,7.0,0.0,0.0,America/Los_Angeles,America/New_York
4,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,22:52,22:47,-5.0,0.0,0.0,-1,2200-2259,19.0,23:06,06:57,15.0,07:29,07:12,-17.0,0.0,0.0,-2,0700-0759,337.0,291.0,2475.0,10,0.0,0.0,0.0,0.0,0.0,America/Los_Angeles,America/New_York


## Check for any missing values

In [12]:
import pyspark.sql.functions as F

In [13]:
tz_cols = ['origin_timezone', 'destination_timezone']

In [14]:
joined_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in tz_cols
]).toPandas()

/opt/venv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

,origin_timezone,destination_timezone
0,0,0


## Load data to data lake

In [15]:
s3_key = "cleaned/Flight_Delay_Prediction_Datasets/flight_data_with_tz/"
write_transformed_data(spark, joined_df, s3_key)